In [0]:
import sys, os

# Ensure the repo root is on the import path
_repo_root = os.path.dirname(os.path.abspath("__file__"))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from config import *
from lseg_api import (
    get_access_token, fetch_reports_info, extract_rows,
    QUERY_BUILDERS, get_retrieval_url,
)
from downloader import BulkDownloader
from ranking import (
    rank_annual_reports, rank_esg_reports,
    get_language_map_df, get_submission_type_df, get_feed_priority_df,
)

from pyspark.sql import functions as F, Window
from datetime import datetime, date

## LSEG Filings Pipeline

Two modes of operation:

1. **Simple mode** — You have a file/list of document IDs → download PDFs directly
2. **Full mode** — Start from a universe CSV → map to LSEG org IDs → fetch metadata → rank → download

Set the `mode` widget below to choose.

In [0]:
dbutils.widgets.dropdown("mode", "simple", ["simple", "full"], "Pipeline mode")
dbutils.widgets.text("doc_ids_path", DOC_IDS_EXCEL_PATH, "Doc IDs file path (simple mode)")
dbutils.widgets.text("universe_csv_path", "", "Universe CSV path (full mode)")
dbutils.widgets.text("output_subfolder", REPORT_SUBFOLDER, "Output blob subfolder")
dbutils.widgets.dropdown("output_target", "local", ["local", "blob"], "Download to")

_mode = dbutils.widgets.get("mode")
_doc_ids_path = dbutils.widgets.get("doc_ids_path")
_universe_csv_path = dbutils.widgets.get("universe_csv_path")
_output_subfolder = dbutils.widgets.get("output_subfolder")
_output_target = dbutils.widgets.get("output_target")

# Resolve the local output directory (project_root/lseg_filings)
_local_output_dir = os.path.join(_repo_root, LOCAL_OUTPUT_SUBFOLDER)

print(f"Mode: {_mode}")
print(f"Output: {_output_target} → {_local_output_dir if _output_target == 'local' else _output_subfolder}")

In [0]:
if _mode == "simple":
    import pandas as pd

    # ── Load doc IDs from Excel / CSV ──────────────────────────────
    if _doc_ids_path:
        if _doc_ids_path.endswith((".xlsx", ".xls")):
            # Excel: copy to local tmp, read with pandas
            _tmp = "/tmp/_lseg_doc_ids.xlsx"
            dbutils.fs.cp(_doc_ids_path, f"file:{_tmp}")
            pdf = pd.read_excel(_tmp)
        else:
            # CSV via Spark
            pdf = spark.read.csv(_doc_ids_path, inferSchema=True, header=True).toPandas()

        print(f"Columns: {list(pdf.columns)}")
        print(f"Rows:    {len(pdf)}")
        display(pdf.head(10))

        # Auto-detect the document ID column
        _id_candidates = [c for c in pdf.columns if "doc" in c.lower() and "id" in c.lower()]
        if _id_candidates:
            _id_col = _id_candidates[0]
        else:
            # Fall back to first integer-like column
            _id_col = next(
                (c for c in pdf.columns if pd.api.types.is_integer_dtype(pdf[c])),
                pdf.columns[0],
            )
        print(f"Using column: {_id_col}")
        doc_ids = pdf[_id_col].dropna().astype(int).tolist()
    else:
        doc_ids = []  # Add IDs inline if no file

    print(f"\nLoaded {len(doc_ids)} document IDs")

    # ── Download ──────────────────────────────────────────────────
    if doc_ids:
        dl = BulkDownloader(dbutils)
        if _output_target == "local":
            failed = dl.download_to_local(doc_ids, output_dir=_local_output_dir)
        else:
            failed = dl.download_by_ids(doc_ids, subfolder=_output_subfolder)
        print(f"\nFailed IDs ({len(failed)}): {failed}")
    else:
        print("No document IDs to download. Set doc_ids_path or add IDs inline.")

## Full Pipeline Mode

The cells below run the complete pipeline:
1. Load universe CSV and map Robeco Party IDs → LSEG organization IDs
2. Fetch document metadata from the LSEG GraphQL API
3. Rank and deduplicate to pick the best report per org
4. Bulk download the selected PDFs

Skip these cells if using simple mode.

In [0]:
if _mode == "full":
    assert _universe_csv_path, "Set the universe_csv_path widget for full mode"

    universe_raw = spark.read.csv(_universe_csv_path, inferSchema=True, header=True)
    universe = (
        universe_raw
        .select(
            F.col("Robeco Party ID").cast("long").alias("robeco_party"),
            F.lit(1).alias("si_universe"),
            "company_name",
        )
        .where(F.col("robeco_party").isNotNull())
    )
    print(f"Universe: {universe.count()} parties")
    universe.display()

In [0]:
if _mode == "full":
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from time import monotonic
    from downloader import RateLimiter, TokenProvider

    _today = date.today()
    _from_3y = date(_today.year - 3, _today.month, 1)

    rate_limiter = RateLimiter(REQUESTS_PER_SECOND)
    token_provider = TokenProvider(dbutils)

    def fetch_org_metadata(org_id, mid_category, from_date, to_date):
        """Fetch filing metadata for one org + category."""
        rate_limiter.wait()
        token = token_provider.get()
        query_builder = QUERY_BUILDERS[mid_category]
        query = query_builder(org_id, from_date, to_date)
        response = fetch_reports_info(token, query)
        return extract_rows(org_id, response)

    # Collect org IDs (assumes universe has been mapped to LSEG org IDs)
    # TODO: Add ISIN-based mapping step if your universe doesn't have organization_id
    org_ids = [row["robeco_party"] for row in universe.select("robeco_party").collect()]

    all_rows = []
    for mid_cat in [1, 26]:  # Annual, ESG
        print(f"\nFetching category {mid_cat} for {len(org_ids)} organizations...")
        started = monotonic()
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(fetch_org_metadata, oid, mid_cat, _from_3y, _today): oid
                for oid in org_ids
            }
            for future in as_completed(futures):
                oid = futures[future]
                try:
                    rows = future.result()
                    all_rows.extend(rows)
                except Exception as e:
                    print(f"  Failed org {oid}: {e}")
        print(f"  Done in {monotonic() - started:.1f}s, {len(all_rows)} rows total")

    doc_meta_df = spark.createDataFrame(all_rows)
    print(f"\nTotal metadata rows: {doc_meta_df.count()}")
    doc_meta_df.display()

In [0]:
if _mode == "full":
    ranked_annual = rank_annual_reports(doc_meta_df.where(F.col("midlevelcategoryid") == 1), spark)
    ranked_esg = rank_esg_reports(doc_meta_df.where(F.col("midlevelcategoryid") == 26), spark)

    to_download = (
        ranked_annual.select("organization_id", "document_id", "midlevelcategoryid")
        .unionByName(
            ranked_esg.select("organization_id", "document_id", "midlevelcategoryid")
        )
    )

    print(f"Annual: {ranked_annual.count()}, ESG: {ranked_esg.count()}")
    print(f"Total to download: {to_download.count()}")

In [0]:
if _mode == "full":
    language_map = get_language_map_df(spark)

    reports = (
        to_download
        .join(doc_meta_df, ["organization_id", "document_id"])
        .join(language_map, "languageid", "left")
        .join(universe, F.col("organization_id") == F.col("robeco_party"), "left")
        .rdd.map(lambda row: row.asDict())
        .collect()
    )

    print(f"Downloading {len(reports)} reports...")
    dl = BulkDownloader(dbutils)
    failed = dl.download_reports(reports, subfolder=_output_subfolder)
    print(f"\nFailed: {failed}")

## Publish to `sistrategist`

Downstream consumers can only read the `sistrategist` container. These cells copy the PDFs from the working container (`aisearch` or `datascientist`) to `sistrategist`, keeping sub-folders and blob metadata.

* `publish = dry_run` (default) lists what would be copied, without writing anything.
* `publish = execute` copies. Existing targets are skipped, never overwritten. The source is read-only.
* Use `publish_limit` for a small trial run first.
* The manifest is shown below and saved to `aisearch/_publish_manifests/`.

In [0]:
dbutils.widgets.dropdown("publish", "dry_run", ["skip", "dry_run", "execute"], "Publish to sistrategist")
dbutils.widgets.dropdown("publish_source_container", STORAGE_CONTAINER, list(PUBLISH_ALLOWED_SOURCES), "Publish source container")
dbutils.widgets.text("publish_dest_prefix", PUBLISH_SUBFOLDER, "Publish destination prefix")
dbutils.widgets.text("publish_limit", "", "Publish limit (blank = all)")

_publish = dbutils.widgets.get("publish")
_publish_source_container = dbutils.widgets.get("publish_source_container")
_publish_dest_prefix = dbutils.widgets.get("publish_dest_prefix")
_publish_limit = int(dbutils.widgets.get("publish_limit")) if dbutils.widgets.get("publish_limit").strip() else None

print(
    f"Publish: {_publish} | {_publish_source_container}/{_output_subfolder} -> "
    f"{PUBLISH_CONTAINER}/{_publish_dest_prefix} | limit={_publish_limit}"
)

In [0]:
if _publish != "skip":
    from publisher import publish_pdfs, manifest_csv, manifest_filename
    from dataclasses import asdict

    publish_rows = publish_pdfs(
        source_container=_publish_source_container,
        source_prefix=_output_subfolder,
        dest_prefix=_publish_dest_prefix,
        execute=(_publish == "execute"),
        limit=_publish_limit,
        dbutils=dbutils,
    )

    if publish_rows:
        spark.createDataFrame([asdict(r) for r in publish_rows]).display()

    if _publish == "execute":
        _manifest_path = storage_base_url("_publish_manifests/") + manifest_filename()
        dbutils.fs.put(_manifest_path, manifest_csv(publish_rows), overwrite=False)
        print(f"Manifest saved to {_manifest_path}")
else:
    print("Publishing skipped")